# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the [FAIR² dataset package](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

This dataset contains clinical and pathological variables for 77 cancer survivors who developed second primary colorectal cancer, including demographics, comorbidities, details of first and second primary cancer, treatment history, diagnosis intervals, anatomical location, histopathology, distant metastasis, and microsatellite instability (MSI-H) status.


In [ ]:
# Install mlcroissant if not present
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Set the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n{metadata.description}")

## 2. Data Overview
Inspect available record sets and their fields. All references will be made using their `@id` values from the Croissant schema.

In [ ]:
# Find the available record sets in the dataset
from pprint import pprint

# Explore the record sets
if hasattr(dataset.metadata, 'record_sets'):
    record_sets = dataset.metadata.record_sets
else:
    # For some Croissant datasets, use get_record_sets()
    record_sets = list(dataset.record_sets())

record_set_ids = []
print("Available record sets (by @id):\n-----------------------------")
for rs in dataset.record_sets():
    rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else (rs['id'] if 'id' in rs else str(rs))
    record_set_ids.append(rs_id)
    name = rs.get('name', 'N/A') if isinstance(rs, dict) else getattr(rs, 'name', 'N/A')
    print(f"  - {rs_id}: {name}")

# For each record set, show available fields and their @ids
print("\nFields within each record set:")
for rs in dataset.record_sets():
    rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else (rs['id'] if 'id' in rs else str(rs))
    print(f"\nRecord set: {rs_id}")
    try:
        fields = rs['fields'] if isinstance(rs, dict) and 'fields' in rs else getattr(rs, 'fields', None)
        if not fields:
            # Try fetching via Croissant API (records method detection)
            sample = next(dataset.records(record_set=rs_id))
            print(f"  Available fields: {list(sample.keys())}")
        else:
            for f in fields:
                f_id = f['@id'] if isinstance(f, dict) and '@id' in f else (f['id'] if 'id' in f else str(f))
                name = f.get('name', 'N/A') if isinstance(f, dict) else getattr(f, 'name', 'N/A')
                print(f"    - {f_id}: {name}")
    except Exception as e:
        print(f"  [Could not get fields: {e}]")

## 3. Data Extraction
Load data from record sets with their record set and field `@id` values.

In [ ]:
# Extract data for each record set and store as DataFrames
# Using the @id values obtained above
available_record_sets = record_set_ids
dataframes = {}

for rs_id in available_record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records from record set '{rs_id}'. Columns: {df.columns.tolist()}")
        else:
            print(f"Record set '{rs_id}' is empty or not accessible.")
    except Exception as e:
        print(f"Error loading data from '{rs_id}': {e}")

# As an example, pick the first record set to explore:
if dataframes:
    selected_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in record set '{selected_rs_id}':")
    print(dataframes[selected_rs_id].columns.tolist())
    dataframes[selected_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Perform data processing using field `@id`s, such as filtering, normalizing, and grouping. Replace `<numeric_field_id>` and `<group_field>` using the IDs found above.

In [ ]:
# Conduct EDA on a numeric field
rs_id = selected_rs_id
df = dataframes[rs_id].copy()

# Try to identify a numeric field automatically (commonly 'age', 'interval', or 'metastasis_count', etc.)
numeric_fields = [c for c in df.columns if df[c].dtype in (np.float64, np.int64) or df[c].apply(lambda x: isinstance(x, (int, float))).all()]
if not numeric_fields:
    # Try to coerce some fields to numeric
    for c in df.columns:
        try:
            converted = pd.to_numeric(df[c], errors='raise')
            numeric_fields.append(c)
        except Exception:
            continue

if numeric_fields:
    numeric_field = numeric_fields[0]
    # If field has @id format, use as is; otherwise, mention name
    print(f"Using numeric field for EDA: {numeric_field}")
else:
    print("No numeric fields found in this record set.")

# Filter for values above a threshold (10 as example)
threshold = 10
try:
    # Attempt conversion if necessary
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with '{numeric_field}' > {threshold} (n={len(filtered_df)}):")
    print(filtered_df.head())

    # Normalization
    field_norm = f"{numeric_field}_normalized"
    filtered_df[field_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, field_norm]].head())

    # Try grouping by another categorical field (e.g., by anatomical location, sex, etc.)
    possible_group_fields = [c for c in df.columns if df[c].dtype == 'object' and c != numeric_field]
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean '{numeric_field}' by '{group_field}':")
        print(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
except Exception as e:
    print(f"EDA error: {e}")

## 5. Visualization
Visualize the distribution and relationships of key fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Visualize the numeric field's distribution
if numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True, color='skyblue')
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouping categorical exists
    if group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=40)
        plt.show()
else:
    print("No numeric fields found for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a clinical dataset defined via a Croissant schema using `mlcroissant`.

- We loaded metadata and inspected available record sets and their fields using their `@id`s.
- We extracted the main record set into a DataFrame for pandas-based analysis.
- We identified a numeric clinical field, filtered and normalized it, grouped by a categorical attribute, and visualized the distributions.

This approach allows robust and reproducible exploration of FAIR-compliant biomedical datasets.
